# 12 — Durable Silver normalization and controlled legacy preservation

Reconstructs only validated durable Bronze through the existing backend adapter and SourceEpochs.
Run the complete compatibility report first. Migration is disabled by default; only an exactly recognized legacy contract can be preserved under a deterministic backup name before canonical creation.
Unknown schemas and backup conflicts stop the whole plan. No lossy backfill or legacy input is permitted.
An exclusive maintenance window (no other catalog/data writers) is required for migration and reruns. Catalog DDL is not a multi-table transaction. Backups are read-only to this pipeline; an administrator must enforce read-only access for other principals.


In [ ]:
# Resolve shared code from the existing notebook in this kernel.
import os
import sys
import json
from pathlib import Path
roots = [Path(os.environ["AIDP_PROJECT_ROOT"])] if os.getenv("AIDP_PROJECT_ROOT") else [Path.cwd(), *Path.cwd().parents]
root = next((p for p in roots if (p / "notebooks_aidp/10_event_capture_foundation.ipynb").is_file()), None)
if root is None:
    raise RuntimeError("Set AIDP_PROJECT_ROOT to the project checkout in Workbench.")
os.environ["AIDP_PROJECT_ROOT"] = str(root.resolve())
foundation = json.loads((root / "notebooks_aidp/10_event_capture_foundation.ipynb").read_text())
for cell in foundation["cells"]:
    if "aidp_shared" in cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(cell["source"]), "10_event_capture_foundation", "exec"), globals())
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Explicit operator approval only. Never read this flag from environment variables.
ALLOW_LEGACY_SILVER_MIGRATION = False

# Only the exact schema observed in Workbench is authorized as known legacy.
LEGACY_LAPS_DDL = """session_id STRING, participant_id STRING, observation_id STRING,
lap_number INT, lap_time_seconds DOUBLE, sector_1_seconds DOUBLE,
sector_2_seconds DOUBLE, sector_3_seconds DOUBLE, position INT,
gap_to_max_seconds DOUBLE, gap_to_leader_seconds DOUBLE, source STRING,
observed_at TIMESTAMP, operator_note STRING, created_at TIMESTAMP,
ingested_at TIMESTAMP, operational_context STRING, evidence_domain STRING,
event_contract_version STRING, source_message_id STRING, source_timestamp TIMESTAMP,
recorded_at TIMESTAMP, idempotency_key STRING, delivery_id STRING,
silver_processed_at TIMESTAMP"""
KNOWN_LEGACY_SILVER = {"event_laps": LEGACY_LAPS_DDL}
LEGACY_RECEIPT = "aidp.durable.legacy_preservation.v1"

def schema_fields(schema):
    return [(f.name, f.dataType.simpleString()) for f in schema]

def same_contract(actual, expected):
    return len(actual) == len(expected) and dict(actual) == dict(expected)

def silver_snapshot(spark, target):
    if not spark.catalog.tableExists(target):
        return None
    frame = spark.table(target)
    # Metadata only: no row values or participant identities are displayed.
    detail = spark.sql(f"DESCRIBE DETAIL {target}").first().asDict()
    return {"schema": schema_fields(frame.schema), "schema_json": frame.schema.json(),
            "rows": frame.count(), "id": detail.get("id"),
            "format": detail.get("format"), "location": detail.get("location"),
            "receipt": (detail.get("properties") or {}).get(LEGACY_RECEIPT)}

def inspect_silver_contracts(spark):
    report = []
    # Inspect all ten targets, including all five priority contracts, before DDL.
    for name, ddl in SILVER_TABLES.items():
        target = table(SILVER_SCHEMA, name)
        backup = target + "_legacy_pre_durable"
        expected = schema_fields(spark.createDataFrame([], ddl).schema)
        legacy = (schema_fields(spark.createDataFrame([], KNOWN_LEGACY_SILVER[name]).schema)
                  if name in KNOWN_LEGACY_SILVER else None)
        item = {"table": target, "backup": backup, "expected_schema": expected,
                "known_legacy_schema": legacy}
        try:
            current = silver_snapshot(spark, target)
            preserved = silver_snapshot(spark, backup)
            item.update(current=current, preserved=preserved,
                        backup_exists=preserved is not None,
                        classification=("MISSING" if current is None else
                            "COMPATIBLE" if same_contract(current["schema"], expected)
                            else "LEGACY_INCOMPATIBLE"),
                        recognized_legacy=bool(current and legacy and
                            same_contract(current["schema"], legacy)))
        except Exception as exc:
            # Complete the report even when an individual metadata read fails.
            item.update(classification="LEGACY_INCOMPATIBLE", inspection_error=type(exc).__name__)
        report.append(item)
    for item in report:
        print({"silver_compatibility": item})
    return report

def preservation_receipt(item, snapshot):
    return {"canonical": item["table"], "backup": item["backup"],
            "schema_sha256": digest(snapshot["schema_json"]),
            "rows": snapshot["rows"], "delta_id": snapshot["id"]}

def verify_preserved(item, snapshot):
    require(snapshot is not None, "Missing preserved table: " + item["backup"])
    require(item["known_legacy_schema"] is not None and
            same_contract(snapshot["schema"], item["known_legacy_schema"]),
            "Unknown backup schema: " + item["backup"])
    require(snapshot["format"] == "delta" and bool(snapshot["id"]),
            "Backup is not an identified Delta table: " + item["backup"])
    require(snapshot["receipt"] == packed(preservation_receipt(item, snapshot)),
            "Backup preservation receipt missing or mismatched: " + item["backup"])

def plan_silver_migration(report, allow=False):
    require(type(allow) is bool, "Migration approval must be a Python boolean")
    actions, errors = [], []
    for item in report:
        if "inspection_error" in item:
            errors.append("Metadata inspection failed: " + item["table"])
            continue
        current, backup = item["current"], item["preserved"]
        if backup is not None:
            try:
                verify_preserved(item, backup)
                if current is not None and item["classification"] == "COMPATIBLE":
                    require(current["id"] != backup["id"] and
                            current["location"] != backup["location"],
                            "Canonical aliases preserved storage: " + item["table"])
            except RuntimeError as exc:
                errors.append(str(exc))
        if item["classification"] == "LEGACY_INCOMPATIBLE":
            if backup is not None:
                errors.append("CONFLICT canonical incompatible + backup exists: " + item["table"])
            elif not item["recognized_legacy"]:
                errors.append("Unrecognized legacy contract: " + item["table"])
            elif current["format"] != "delta" or not current["id"]:
                errors.append("Migration requires an identified Delta table: " + item["table"])
            elif not allow:
                errors.append("Approval required: " + item["table"])
            elif current["receipt"] not in (None, packed(preservation_receipt(item, current))):
                errors.append("CONFLICT existing migration receipt: " + item["table"])
            else:
                actions.append(("RENAME_CREATE", item))
        elif current is None:
            if backup is not None and not allow:
                errors.append("Approval required to resume migration: " + item["table"])
            else:
                actions.append(("CREATE", item))
    require(not errors, "Silver preflight stopped without writes:\n" + "\n".join(errors))
    return actions

def apply_silver_migration(spark, report, allow=False):
    actions = plan_silver_migration(report, allow)
    # Fail before any mutation if any table changed since the printed preflight.
    for item in report:
        require(silver_snapshot(spark, item["table"]) == item["current"] and
                silver_snapshot(spark, item["backup"]) == item["preserved"],
                "Concurrent catalog/data change: " + item["table"])
    for action, item in actions:
        target, backup = item["table"], item["backup"]
        if action == "RENAME_CREATE":
            receipt = packed(preservation_receipt(item, item["current"]))
            # Persist pre-rename counts/schema/id so an interrupted run can be
            # resumed without inventing a historical row count. No row is written.
            escaped = receipt.replace("'", "''")
            spark.sql(f"ALTER TABLE {target} SET TBLPROPERTIES ('{LEGACY_RECEIPT}' = '{escaped}')")
            spark.sql(f"ALTER TABLE {target} RENAME TO {backup}")
            preserved = silver_snapshot(spark, backup)
            verify_preserved(item, preserved)
            require(preserved["id"] == item["current"]["id"] and
                    preserved["schema_json"] == item["current"]["schema_json"] and
                    preserved["rows"] == item["current"]["rows"],
                    "Post-rename preservation check failed: " + backup)
            require(not spark.catalog.tableExists(target), "Canonical still exists after rename")
        # Default creation only; no REPLACE, overwrite, legacy SELECT or backfill.
        name = target.rsplit(".", 1)[1]
        check_schema(spark, target, SILVER_TABLES[name], create=True)
        created = silver_snapshot(spark, target)
        require(created["rows"] == 0, "New canonical table must be empty: " + target)
        preserved = silver_snapshot(spark, backup)
        if preserved is not None:
            verify_preserved(item, preserved)
            require(created["id"] != preserved["id"] and
                    created["location"] != preserved["location"],
                    "Canonical aliases preserved storage: " + target)
    for item in report:
        check_schema(spark, item["table"], SILVER_TABLES[item["table"].rsplit(".", 1)[1]])

def report_legacy_preservation(spark, report):
    result = []
    for item in report:
        backup = silver_snapshot(spark, item["backup"])
        if backup is not None:
            verify_preserved(item, backup)
            baseline = item["preserved"] or item["current"]
            require(baseline is not None and backup["id"] == baseline["id"] and
                    backup["schema_json"] == baseline["schema_json"] and
                    backup["rows"] == baseline["rows"],
                    "Preserved legacy evidence changed: " + item["backup"])
            result.append({"table": item["backup"],
                           "legacy_pre_migration_rows": json.loads(backup["receipt"])["rows"],
                           "legacy_post_migration_rows": backup["rows"]})
    print({"legacy_preservation": result})
    return result


In [ ]:
silver_compatibility = inspect_silver_contracts(spark)
# This cell performs no mutation. Review every entry, including unknown schemas.
silver_migration_plan = plan_silver_migration(silver_compatibility, ALLOW_LEGACY_SILVER_MIGRATION)
print({"migration_actions": [(action, item["table"]) for action, item in silver_migration_plan]})


In [ ]:
segments, raw_frames = bronze_input(spark)
# Reconcile Bronze with the archive before any downstream processing.
archive_segments, archive_frames = read_archive()
equal_rows(spark, table(BRONZE_SCHEMA, "event_segments"), archive_segments, SEGMENT_DDL, ["logical_session_id", "segment_id"])
equal_rows(spark, table(BRONZE_SCHEMA, "event_raw_frames"), archive_frames, RAW_DDL, ["logical_session_id", "segment_id", "sequence"])
events, coverage = normalize(segments, raw_frames)
require(events, "No factual events parsed")
require(EXPECTED_CHANNELS <= set(coverage), "Missing expected channels: " + str(sorted(EXPECTED_CHANNELS - set(coverage))))
outputs = silver_rows(events)
apply_silver_migration(spark, silver_compatibility, ALLOW_LEGACY_SILVER_MIGRATION)
for name, rows in outputs.items():
    keys = ["logical_session_id", "source_epoch", "event_sequence"]
    if SILVER_TABLES[name] == ROW_DDL:
        keys.append("row_index")
    merge_rows(spark, table(SILVER_SCHEMA, name), rows, SILVER_TABLES[name], keys, "content_hash")
    equal_rows(spark, table(SILVER_SCHEMA, name), rows, SILVER_TABLES[name], keys)
print({"parser_errors": 0, "channels": coverage, "normalized_events": len(events)})


In [ ]:
# Aggregates only: never display participant names or raw event payloads.
silver_validation = []
for name, ddl in SILVER_TABLES.items():
    target = table(SILVER_SCHEMA, name)
    frame = spark.table(target)
    keys = ["logical_session_id", "source_epoch", "event_sequence"] + (["row_index"] if ddl == ROW_DDL else [])
    duplicates = frame.groupBy(*keys).count().filter("count > 1").count()
    require(duplicates == 0, "Duplicate natural keys: " + target)
    bounds = frame.agg(F.min("event_sequence").alias("min_event_sequence"),
                       F.max("event_sequence").alias("max_event_sequence")).first().asDict()
    silver_validation.append({"table": target, "row_count": frame.count(),
        "logical_sessions": frame.select("logical_session_id").distinct().count(),
        "source_epochs": frame.select("logical_session_id", "source_epoch").distinct().count(),
        "duplicate_natural_keys": duplicates, **bounds})
print({"silver": silver_validation, "parser_errors": 0, "channel_coverage": coverage,
       "source_epoch_count": len({(e["logical_session_id"], e["source_epoch"]) for e in events})})
legacy_validation = report_legacy_preservation(spark, silver_compatibility)
